# Réglage de hyperparamètres

Vocabulaire : les paramètres des modèles de machine learning que l’on peut modifier sont appelés des hyperparamètres. Attention à ne pas les confondre avec les paramètres du modèle qui eux sont calculés automatiquement pendant l’entraînement.

Exemple : le nombre de couches d’un réseau de neurones est un hyperparamètre, le biais d’un neurone donné est un paramètre du réseau.

Méthodes de réglage des hyperparamètres: Gridsearch, Random search, etc.

https://larevueia.fr/3-methodes-pour-optimiser-les-hyperparametres-de-vos-modeles-de-machine-learning/


Dans cette étude, nous allons regarder un exemple de réglage pour la méthode SVC. La base de données: Titanic!

0. Importer des librairies:
- pandas : Utilisé pour la manipulation et l'analyse des données
- train_test_split : Bibliothèque Sklearn pour diviser des tableaux ou des matrices en sous-ensembles aléatoires de formation et de test.
- GridSearchCV : Bibliothèque Sklearn permettant d'effectuer une recherche exhaustive sur des valeurs de paramètres spécifiées pour un estimateur.
- RandomizedSearchCV : Bibliothèque Sklearn pour effectuer une recherche aléatoire sur les hyper paramètres.
- svm : Bibliothèque Sklearn Support Vector Machines

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV,RandomizedSearchCV
from sklearn import svm

1. Charger la database "titanic.csv"


In [2]:
df = pd.read_csv('titanic.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


2. Prétraitement de la database (par exemple retirer les données sans étiquette "Survived" ou les variables inutiles)

Dans cette étude, nous ne retiendrons que les variables quantitatives

In [4]:
# Remove rows with missing target values
df.dropna(axis=0, subset=['Survived'], inplace=True)
y = df.Survived # Target variable             
df.drop(['Survived'], axis=1, inplace=True) # Removing target variable from training data

df.drop(['Age'], axis=1, inplace=True) # Remove columns with null values

# Select numeric columns only
numeric_cols = [cname for cname in df.columns if df[cname].dtype in ['int64', 'float64']]
X = df[numeric_cols].copy()

print("Shape of input data: {} and shape of target variable: {}".format(X.shape, y.shape))

# X.head() 
pd.concat([X,y], axis=1).head()# Show first 5 training examples

Shape of input data: (891, 5) and shape of target variable: (891,)


,PassengerId,Pclass,SibSp,Parch,Fare,Survived
0,1,3,1,0,7.2500,0
1,2,1,1,0,71.2833,1
2,3,3,0,0,7.9250,1
3,4,1,1,0,53.1000,1
4,5,3,0,0,8.0500,0


2. Analyser la base de données 

Vous êtes libres à donner la réponse. Suggestions: afficher par exemple les statistiques de données, hitogramme, pairplot...

## Sans réglage des hyperparamètres

3. Faire "train_test_split" 

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42)
print('X_train dimension= ', X_train.shape)
print('X_test dimension= ', X_test.shape)
print('y_train dimension= ', y_train.shape)
print('y_train dimension= ', y_test.shape)

X_train dimension=  (712, 5)
X_test dimension=  (179, 5)
y_train dimension=  (712,)
y_train dimension=  (179,)


4. Fit le modèle SVC, afficher le score sur "test"

In [6]:
clf= svm.SVC()
clf.fit(X_train, y_train)
print('Model score using default parameters is = ', clf.score(X_test, y_test))

Model score using default parameters is =  0.5977653631284916


## Réglage des hyperparamètres

5. Quels sont les hyperparamètres d'un modèle SVC?
[https://scikit-learn.org/stable/auto_examples/svm/plot_svm_kernels.html#sphx-glr-auto-examples-svm-plot-svm-kernels-py](https://scikit-learn.org/stable/auto_examples/svm/plot_svm_kernels.html#sphx-glr-auto-examples-svm-plot-svm-kernels-py)

Votre réponse:

6. Créer un grid pour ces hyperparamètres et lancer GridSearchCV

In [7]:
# Let create parameter grid for GridSearchCV
parameters = {  'C':[0.01, 1, 5],             # regularisation = moins sensible au bruit
                'kernel':('linear', 'rbf'),   # type de frontière
                'gamma' :('scale', 'auto')    # spécifique à RBF
              }

In [8]:
gsc = GridSearchCV(estimator = svm.SVC(), param_grid= parameters,cv= 5,verbose =1)

# Fitting the model for grid search. It will first find the best parameter combination using cross validation. 
# Once it has the best combination, it runs fit again on all data passed to fit (without cross-validation), 
# to built a single new model using the best parameter setting.
gsc.fit(X_train, y_train) 

Fitting 5 folds for each of 12 candidates, totalling 60 fits


GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.01, 1, 5], 'gamma': ('scale', 'auto'),
                         'kernel': ('linear', 'rbf')},
             verbose=1)

7. Afficher les meilleurs hyperparamètres et le score correspondant

In [9]:
print(f'Best hyperparameters: {gsc.best_params_}') 
print(f'Best score: {gsc.best_score_}')
print('Detailed GridSearchCV result is as below')
gsc_result = pd.DataFrame(gsc.cv_results_).sort_values('mean_test_score',ascending= False)
gsc_result[['param_C','param_kernel','param_gamma','mean_test_score']]

Best hyperparameters: {'C': 0.01, 'gamma': 'scale', 'kernel': 'linear'}
Best score: 0.6825470304343544
Detailed GridSearchCV result is as below


,param_C,param_kernel,param_gamma,mean_test_score
0,0.01,linear,scale,0.682547
2,0.01,linear,auto,0.682547
6,1.00,linear,auto,0.676923
4,1.00,linear,scale,0.676923
9,5.00,rbf,scale,0.672737
8,5.00,linear,scale,0.669891
10,5.00,linear,auto,0.669891
5,1.00,rbf,scale,0.647454
3,0.01,rbf,auto,0.623599
1,0.01,rbf,scale,0.623599


8. Refaire les recherches avec RandomizedSearchCV

In [10]:
# n_iter=5 > Number of parameter settings that are sampled. 
# So instaed of 12 it will randomly search for only 5 combinations for each fold
rsc = RandomizedSearchCV(estimator = svm.SVC(), param_distributions= parameters,cv=5,n_iter = 1,verbose =1)
rsc.fit(X_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits


RandomizedSearchCV(cv=5, estimator=SVC(), n_iter=1,
                   param_distributions={'C': [0.01, 1, 5],
                                        'gamma': ('scale', 'auto'),
                                        'kernel': ('linear', 'rbf')},
                   verbose=1)

9. Afficher les meilleurs hyperparamètres et le score correspondant

In [11]:
print(f'Best hyperparameters: {rsc.best_params_}') 
print(f'Best score: {rsc.best_score_}')
print('Detailed RandomizedSearchCV result is as below')
rsc_result = pd.DataFrame(rsc.cv_results_).sort_values('mean_test_score',ascending= False)
rsc_result[['param_C','param_kernel','param_gamma','mean_test_score']]

Best hyperparameters: {'kernel': 'rbf', 'gamma': 'auto', 'C': 1}
Best score: 0.5996749729144095
Detailed RandomizedSearchCV result is as below


,param_C,param_kernel,param_gamma,mean_test_score
0,1,rbf,auto,0.599675


10. Votre conclusion: